In [1]:
import json

from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from config import (
    CHUNKS_JSON_PATH,
    DEFAULT_EMBEDDING_MODEL_ID,
)
from evaluation.evaluate import Evaluator

/tmp/ipykernel_14240/698318276.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [2]:
chunks = json.loads(CHUNKS_JSON_PATH.read_text(encoding="utf-8"))
docs = [
    Document(
        page_content=chunk["text"],
        metadata={"doc_id": chunk["doc_id"], "source": chunk["source"]},
    )
    for chunk in chunks
]

In [3]:
# Embed
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=HuggingFaceEmbeddings(model_name=DEFAULT_EMBEDDING_MODEL_ID),
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

/home/nt/.cache/pypoetry/virtualenvs/genai-study-gR7JEMom-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2171.42it/s]


In [4]:
for doc in retriever.invoke("What is the length of Vin's hair?"):
    print("**" * 20)
    print(doc.page_content)

****************************************
She yawned, rubbed her temples again, then threw on a robe and made her way to the bathroom. She was pleased to see that Clubs’s apprentices had remembered to draw her a bath. She locked the door, disrobed, and settled into the warm, lightly scented bathwater. Had she ever really found those scents obnoxious? The smell would make her less inconspicuous, true, but that seemed a slim price for ridding herself of the dirt and grime she’d picked up while traveling.

She still found longer hair an annoyance. She washed it, combing out the tangles and knots, wondering how the court women could stand hair that went all the way down their backs. How long must they spend combing and primping beneath a servant’s care? Vin’s hair hadn’t reached her shoulders yet, and she was already loath to let it get longer. It would fly about and whip her face when she jumped, not to mention provide her foes with something to grab on to.
********************************

In [5]:
evaluator = Evaluator(k_values=(1, 3, 5), experiment_name="naive_rag")
metrics = evaluator.evaluate(retriever)

precision@1: 0.1623
recall@1: 0.1623
mrr@1: 0.1623
precision@3: 0.0904
recall@3: 0.2712
mrr@3: 0.2096
precision@5: 0.0654
recall@5: 0.3270
mrr@5: 0.2223
